# RNA Velocity Analysis with scVelo

Pipeline for data where velocyto barcodes map to custom cell IDs:
- Aggregate spliced/unspliced counts from multiple barcodes → one cell_ID
- Transfer existing UMAP / clusters / cell types from annotated `.h5ad`
- Run scVelo on the merged object without recomputing the embedding

## 0. Parameters — change paths here

In [ ]:
# ─── INPUT ───────────────────────────────────────────────────────────────────
# Merged velocyto loom file (spliced / unspliced / ambiguous layers)
LOOM_PATH = "path/to/merged.loom"          # <-- change

# Annotated AnnData with UMAP, clusters, cell types
ANNDATA_PATH = "path/to/annotated.h5ad"    # <-- change

# Mapping table: CSV/TSV with at least two columns — barcode and cell_ID
# Example:
#   barcode,cell_id
#   ACGTACGT-1,cell_8377_N0
#   TTGGCCAA-1,cell_8377_N0
#   GCTAGCTA-1,cell_8412_N0
MAPPING_PATH = "path/to/barcode_to_cellid.csv"  # <-- change
MAPPING_BARCODE_COL = "barcode"    # column name for barcodes in the mapping file
MAPPING_CELLID_COL  = "cell_id"    # column name for cell IDs

# ─── OUTPUT ──────────────────────────────────────────────────────────────────
OUT_DIR = "results_velocity"

# ─── COLUMN NAMES IN ANNOTATED ANNDATA ───────────────────────────────────────
# obs column with cell type labels
CELLTYPE_KEY = "cell_type"     # change if your column has a different name
# obs column with cluster labels (can be the same as CELLTYPE_KEY)
CLUSTER_KEY  = "leiden"

# ─── scVelo SETTINGS ─────────────────────────────────────────────────────────
VELOCITY_MODE     = "dynamical"   # "stochastic" or "dynamical"
N_HVG             = 2000
MIN_SHARED_COUNTS = 20
SEED              = 42

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import scanpy as sc
import scvelo as scv
import matplotlib.pyplot as plt

sc.settings.verbosity = 2
scv.settings.verbosity = 2
scv.set_figure_params("scvelo", dpi=100, dpi_save=200)

os.makedirs(OUT_DIR, exist_ok=True)
print(f"scVelo {scv.__version__} | Scanpy {sc.__version__}")

## 2. Load loom file

In [ ]:
adata_loom = scv.read(LOOM_PATH, cache=True)
print("Loom:", adata_loom)
print("Layers:", list(adata_loom.layers.keys()))
print("\nFirst 5 barcodes:", adata_loom.obs_names[:5].tolist())

## 3. Load mapping table and aggregate counts by cell_ID

Multiple barcodes that belong to the same cell_ID are **summed** (spliced and unspliced separately).
Barcodes absent from the mapping table are discarded.

In [ ]:
mapping = pd.read_csv(MAPPING_PATH)
print(f"Mapping table: {len(mapping)} rows, columns: {mapping.columns.tolist()}")
mapping = mapping[[MAPPING_BARCODE_COL, MAPPING_CELLID_COL]].drop_duplicates()
mapping = mapping.set_index(MAPPING_BARCODE_COL)

# Keep only barcodes present in both the loom and the mapping table
common_barcodes = adata_loom.obs_names.intersection(mapping.index)
print(f"\nBarcodes in loom:    {adata_loom.n_obs}")
print(f"Barcodes in mapping: {len(mapping)}")
print(f"Overlap:             {len(common_barcodes)}")
print(f"Discarded (no mapping): {adata_loom.n_obs - len(common_barcodes)}")

adata_sub = adata_loom[common_barcodes].copy()
adata_sub.obs["cell_id"] = mapping.loc[common_barcodes, MAPPING_CELLID_COL].values

In [ ]:
def aggregate_by_cell_id(adata, cell_id_col="cell_id"):
    """Sum spliced / unspliced / ambiguous counts across barcodes of the same cell."""
    cell_ids = adata.obs[cell_id_col]
    unique_ids = cell_ids.unique()
    n_cells = len(unique_ids)
    n_genes = adata.n_vars

    # Build a (cells × barcodes) aggregation matrix
    cell_id_index = pd.Categorical(cell_ids, categories=unique_ids)
    agg_matrix = sp.csr_matrix(
        (np.ones(adata.n_obs), (cell_id_index.codes, np.arange(adata.n_obs))),
        shape=(n_cells, adata.n_obs),
    )

    layers_agg = {}
    for layer in adata.layers:
        mat = adata.layers[layer]
        if sp.issparse(mat):
            layers_agg[layer] = (agg_matrix @ mat).astype(np.float32)
        else:
            layers_agg[layer] = (agg_matrix @ mat).astype(np.float32)

    # Use spliced as X
    X = layers_agg.get("spliced", list(layers_agg.values())[0])

    adata_agg = ad.AnnData(
        X=X,
        obs=pd.DataFrame(index=unique_ids),
        var=adata.var.copy(),
        layers=layers_agg,
    )
    adata_agg.obs_names.name = None
    return adata_agg


adata_vel = aggregate_by_cell_id(adata_sub, cell_id_col="cell_id")

# Barcode-to-cell stats
bc_per_cell = adata_sub.obs.groupby("cell_id").size()
print(f"\nAggregated: {adata_vel.n_obs} cells × {adata_vel.n_vars} genes")
print(f"Barcodes per cell — median: {bc_per_cell.median():.0f}, "
      f"max: {bc_per_cell.max()}, min: {bc_per_cell.min()}")
scv.pl.proportions(adata_vel)

## 4. Load annotated AnnData and transfer embedding + annotations

In [ ]:
adata_ann = sc.read_h5ad(ANNDATA_PATH)
print("Annotated AnnData:", adata_ann)
print("obs columns:", adata_ann.obs.columns.tolist())

In [ ]:
# Intersect on cell_IDs present in both objects
common_cells = adata_vel.obs_names.intersection(adata_ann.obs_names)
print(f"Cells in velocity object:  {adata_vel.n_obs}")
print(f"Cells in annotated object: {adata_ann.n_obs}")
print(f"Intersection:              {len(common_cells)}")
print(f"Dropped (no velocity):     {adata_ann.n_obs - len(common_cells)}")

adata_vel = adata_vel[common_cells].copy()
adata_ref = adata_ann[common_cells].copy()   # same order

# ── Transfer UMAP and PCA coordinates ────────────────────────────────────────
if "X_umap" in adata_ref.obsm:
    adata_vel.obsm["X_umap"] = adata_ref.obsm["X_umap"]
if "X_pca" in adata_ref.obsm:
    adata_vel.obsm["X_pca"] = adata_ref.obsm["X_pca"]

# ── Transfer neighbor graph (needed for scv.pp.moments) ──────────────────────
if "neighbors" in adata_ref.uns:
    adata_vel.uns["neighbors"] = adata_ref.uns["neighbors"]
if "connectivities" in adata_ref.obsp:
    adata_vel.obsp["connectivities"] = adata_ref.obsp["connectivities"]
if "distances" in adata_ref.obsp:
    adata_vel.obsp["distances"] = adata_ref.obsp["distances"]

# ── Transfer cell annotations ─────────────────────────────────────────────────
cols_to_transfer = [CELLTYPE_KEY, CLUSTER_KEY]
for col in cols_to_transfer:
    if col in adata_ref.obs.columns:
        adata_vel.obs[col] = adata_ref.obs[col].values
    else:
        print(f"  Warning: column '{col}' not found in annotated AnnData")

print("\nTransfer complete.")
print(adata_vel)

In [ ]:
# Quick sanity check — plot the transferred UMAP with cell types
color_key = CELLTYPE_KEY if CELLTYPE_KEY in adata_vel.obs.columns else CLUSTER_KEY

sc.pl.umap(
    adata_vel,
    color=color_key,
    legend_loc="on data",
    title=f"Transferred UMAP — {color_key}",
)

## 5. scVelo preprocessing

We normalize the spliced/unspliced layers and compute moments **using the transferred neighbor graph**,
so the velocity field is consistent with the original embedding.

In [ ]:
scv.pp.filter_and_normalize(
    adata_vel,
    min_shared_counts=MIN_SHARED_COUNTS,
    n_top_genes=N_HVG,
    log=True,
)

# Use existing neighbor graph if available; otherwise recompute from PCA
if "connectivities" in adata_vel.obsp:
    scv.pp.moments(adata_vel, use_rep=None, n_pcs=None)   # uses existing graph
elif "X_pca" in adata_vel.obsm:
    scv.pp.moments(adata_vel, n_pcs=30, n_neighbors=30)
else:
    # Last resort: recompute PCA on velocity-selected genes
    sc.tl.pca(adata_vel, random_state=SEED)
    scv.pp.moments(adata_vel, n_pcs=30, n_neighbors=30)

print(adata_vel)

## 6. RNA velocity

In [ ]:
if VELOCITY_MODE == "dynamical":
    scv.tl.recover_dynamics(adata_vel, n_jobs=-1)
    scv.tl.velocity(adata_vel, mode="dynamical")
elif VELOCITY_MODE == "stochastic":
    scv.tl.velocity(adata_vel, mode="stochastic")
else:
    raise ValueError("VELOCITY_MODE must be 'dynamical' or 'stochastic'")

scv.tl.velocity_graph(adata_vel, n_jobs=-1)

## 7. Visualization on the original UMAP

In [ ]:
# ── 7.1  Velocity stream ──────────────────────────────────────────────────────
scv.pl.velocity_embedding_stream(
    adata_vel,
    basis="umap",
    color=color_key,
    legend_loc="right margin",
    title=f"RNA velocity — {color_key}",
    save=f"{OUT_DIR}/velocity_stream.png",
)

# ── 7.2  Velocity arrows ──────────────────────────────────────────────────────
scv.pl.velocity_embedding(
    adata_vel,
    basis="umap",
    arrow_length=3,
    arrow_size=2,
    color=color_key,
    save=f"{OUT_DIR}/velocity_arrows.png",
)

In [ ]:
# ── 7.3  Confidence and length ────────────────────────────────────────────────
scv.tl.velocity_confidence(adata_vel)

scv.pl.scatter(
    adata_vel,
    color=["velocity_length", "velocity_confidence"],
    cmap="coolwarm",
    perc=[5, 95],
    save=f"{OUT_DIR}/velocity_confidence.png",
)

In [ ]:
# ── 7.4  PAGA velocity graph ──────────────────────────────────────────────────
scv.tl.velocity_pseudotime(adata_vel)

scv.tl.paga(
    adata_vel,
    groups=color_key,
    use_time_prior="velocity_pseudotime",
)

scv.pl.paga(
    adata_vel,
    basis="umap",
    size=50,
    alpha=0.1,
    min_edge_width=2,
    node_size_scale=1.5,
    save=f"{OUT_DIR}/paga_velocity.png",
)

## 8. Velocity pseudotime

In [ ]:
scv.pl.scatter(
    adata_vel,
    color="velocity_pseudotime",
    cmap="gnuplot",
    save=f"{OUT_DIR}/pseudotime.png",
)

## 9. Top velocity genes per cell type

In [ ]:
scv.tl.rank_velocity_genes(adata_vel, groupby=color_key, min_corr=0.3)

df_top = scv.DataFrame(adata_vel.uns["rank_velocity_genes"]["names"]).head(10)
print(df_top)
df_top.to_csv(f"{OUT_DIR}/top_velocity_genes.csv")

top_gene = df_top.iloc[0, 0]
scv.pl.velocity(
    adata_vel,
    var_names=[top_gene],
    color=color_key,
    save=f"{OUT_DIR}/phase_portrait_{top_gene}.png",
)

## 10. (Dynamical) Latent time

In [ ]:
if VELOCITY_MODE == "dynamical":
    scv.tl.latent_time(adata_vel)

    scv.pl.scatter(
        adata_vel,
        color="latent_time",
        color_map="gnuplot",
        size=80,
        save=f"{OUT_DIR}/latent_time.png",
    )

    top_genes = adata_vel.var["fit_likelihood"].sort_values(ascending=False).index[:10]
    scv.pl.heatmap(
        adata_vel,
        var_names=top_genes,
        sortby="latent_time",
        col_color=color_key,
        yticklabels=True,
        n_convolve=100,
        save=f"{OUT_DIR}/heatmap_latent_time.png",
    )
else:
    print("Latent time only available in dynamical mode.")

## 11. Save

In [ ]:
out_path = f"{OUT_DIR}/adata_velocity.h5ad"
adata_vel.write_h5ad(out_path)
print(f"Saved: {out_path}")
print(adata_vel)